In [ ]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | TeP
# v01
# ----------------------------------------------------------------------
"""

### [TeP_01] (some) Selected Solutions

### 1. Purpose

This notebook presents selected solutions to technical problems
and examples supporting the development of TeP_01.

The selected solutions use dataset_A and illustrate relevant aspects of:
- data integration and consolidation
- analytical dataset construction
- feature representation and target definition
- machine learning and model evaluation

The notebook is **not a step-by-step implementation guide**.

Students are expected to develop their own integrated solutions, including:
- dataset_B integration
- model development, and
- deployment

The notebook provides technical support and reusable approaches,
but does not replace the independent development of the assignment.

### 2. Required Completion

Create a copy of this notebook named
- `00_sns_selected_solutions.ipynb`.

All `TODO` tasks must be completed in that copy.


In [ ]:
# ------------------------------------------------------------
# bootstrap the initial PostgreSQL database
# ------------------------------------------------------------

from pathlib import Path
import subprocess
import sys

# locate the bootstrap script in the current code folder
boot_script = Path("_mLDm_00_sns_db_boot.py").resolve()

# recreate and populate the initial database
subprocess.run([sys.executable, str(boot_script)], check=True)

print("PTS | initial database successfully created")

In [ ]:
# ------------------------------------------------------------
# connect DuckDB to the PostgreSQL database
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the PostgreSQL extension
# INSTALL is required only the first time
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the TeP PostgreSQL database
# its tables will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm_tep user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

# set pg.public as the default schema for subsequent SQL queries
# - allows table names to be used without explicit qualification
db.sql("USE pg.public;")

print("PTS | DuckDB connected to PostgreSQL")

In [ ]:
# ------------------------------------------------------------
# import dataset_A into a PostgreSQL staging table
# ------------------------------------------------------------

# locate dataset_A (2021–2024)
csv_file = Path("../01_input/_mLDm_00_sns_data_A.csv").resolve()

# import all records, preserving repeated observation units
# rename the original Portuguese attributes
db.sql(f"""
-- TODO | identify the staging table name
CREATE OR REPLACE TABLE ... AS
SELECT
  "Período" AS period,
  "Região" AS region,
  "Instituição" AS institution,
  "Código Capítulo Diagnóstico Principal" AS diagnosis_code,
  -- TODO | identify the source and staging attribute
  ... AS diagnosis_description,
  -- TODO | identify the source and staging attribute
  ... AS ...,
  "Sexo" AS sex,
  "Internamentos" AS num_inpatient_admission,
  "Dias de Internamento" AS num_inpatient_day,
  -- TODO | identify the source and staging attribute
  ... AS ...,
  "Óbitos" AS num_death
FROM read_csv(
  '{csv_file.as_posix()}',
  delim=';',
  header=True,
  types={{'Período': 'VARCHAR'}}
);
""")

# verify the imported records
db.sql("""
SELECT COUNT(*) AS n_records
FROM source_stage;
""").show()


# expected output:
# │ n_records │
# │   int64   │
# ├───────────┤
# │    326692 |

In [ ]:
# ------------------------------------------------------------
# inspect the staging table
#
# note:
# pgAdmin can also be used to inspect the database structure
# and explore its contents interactively
# ------------------------------------------------------------

# inspect the staging table schema
db.sql("""
-- TODO | identify the staging table name
DESCRIBE ...;
""").show()

# inspect a sample of source records
db.sql("""
SELECT 
  period,
  -- TODO | identify the staging attribute
  ...,
  -- TODO | identify the staging attribute
  ...,
  -- TODO | identify the staging attribute
  ...,
  -- TODO | identify the staging attribute
  ...,
  age_group,
  sex,
  num_inpatient_admission,
  num_inpatient_day,
  num_ambulatory_episode,
  num_death
-- TODO | identify the staging table name
FROM ...
ORDER BY period ASC
LIMIT 5;
""").show()

In [ ]:
# ------------------------------------------------------------
# establish correspondence with the reference tables
# ------------------------------------------------------------

# Important:
# here, we are using LEFT JOIN so:
# - a NULL value does not necessarily means a NULL value in the table
# - it also means a NULL value in the LEFT JOIN combination

db.sql("""
CREATE OR REPLACE TEMP VIEW v_reference_check AS
SELECT
  s.region,
  s.institution,
  s.diagnosis_code,
  s.diagnosis_description,
  r.id AS id_region,
  i.id AS id_institution,
  d.id AS id_diagnosis_code,
  d.description AS reference_diagnosis_description,
  CASE WHEN s.diagnosis_description = d.description THEN d.id END AS id_diagnosis
FROM source_stage s
LEFT JOIN region r
  ON s.region = r.name
LEFT JOIN institution i
  ON s.institution = i.name
 AND i.id_region = r.id
LEFT JOIN diagnosis d
  ON s.diagnosis_code = d.id;
""")

In [ ]:
# ------------------------------------------------------------
# identify regions without reference correspondence
# ------------------------------------------------------------

db.sql("""
SELECT
  region,
  COUNT(*) AS n_records
FROM v_reference_check
WHERE id_region IS NULL
GROUP BY region
ORDER BY n_records DESC, region;
""").show()

In [ ]:
# ------------------------------------------------------------
# identify institutions without reference correspondence
# ------------------------------------------------------------

db.sql("""
SELECT
  region,
  institution,
  COUNT(*) AS n_records
FROM v_reference_check
-- TODO | identify correspondence condition
WHERE ...
GROUP BY region, institution
ORDER BY n_records DESC, region, institution;
""").show()

In [ ]:
# ------------------------------------------------------------
# identify diagnoses without exact reference correspondence
# ------------------------------------------------------------

db.sql("""
SELECT
  diagnosis_code,
  -- TODO | identify attribute
  ...,
  reference_diagnosis_description,
  -- TODO | count number of unmatched records
  ...
FROM v_reference_check
-- TODO | identify correspondence condition
WHERE ...
GROUP BY diagnosis_code, diagnosis_description, reference_diagnosis_description
ORDER BY n_records DESC, diagnosis_code;
""").show()


# expected output:
# | diagnosis_code │ diagnosis_description │ reference_diagnosis_description │ n_records │
# │     int64      │        varchar        │             varchar             │   int64 

In [ ]:
# ------------------------------------------------------------
# validate source data against reference tables
# ------------------------------------------------------------

# verify exact correspondence with the reference tables
# COUNT(*) counts all records, while COUNT(id) ignores NULL values
# their difference identifies records without a matching entity

db.sql("""
SELECT
  COUNT(*) AS n_records,
  COUNT(*) - COUNT( id_region ) AS n_unmatched_region,
  -- TODO | count number of unmatched records
  ...,
  -- TODO | count number of unmatched records
  ...
FROM v_reference_check;
""").show()


# expected output:
# ┌───────────┬────────────────────┬─────────────────────────┬───────────────────────┐
# │ n_records │ n_unmatched_region │ n_unmatched_institution │ n_unmatched_diagnosis │
# │   int64   │       int64        │          int64          │         int64         │
# ├───────────┼────────────────────┼─────────────────────────┼───────────────────────┤
# │    326692 │                  0 │                       0 │                     0 │
# |___________|____________________|_________________________|_______________________|

In [ ]:
# ------------------------------------------------------------
# apply corrections only to unmatched source values
# ------------------------------------------------------------

# corrections: (attribute, original_value, reference_value)
corrections = [
    # ("institution", "original name", "reference name"),
    # ("diagnosis_description", "original description", "reference description"),
]

# attribute: (matching_attributes, reference_id)
reference_checks = {
    "region":                (["region"], "id_region"),
    "institution":           (["region", "institution"], "id_institution"),
    "diagnosis_code":        (["diagnosis_code"], "id_diagnosis_code"),
    "diagnosis_description": (["diagnosis_code", "diagnosis_description"], "id_diagnosis"),
}

# apply corrections in reference dependency order
applied_corrections = []
total_updated = 0
for attribute, (matching_attributes, reference_id) in reference_checks.items():
    for correction_attribute, original_value, reference_value in corrections:
        if correction_attribute != attribute: continue

        source_columns = []
        check_columns = []
        for matching_attribute in matching_attributes:
            source_columns.append( f"source.{matching_attribute}" )
            check_columns.append( matching_attribute )
            
        source_attributes = ", ".join( source_columns )
        check_attributes = ", ".join( check_columns )
        db.execute(f"""
            UPDATE source_stage AS source
            SET {attribute} = ?
            WHERE source.{attribute} = ?
              AND ({source_attributes}) IN (
                  SELECT {check_attributes}
                  FROM v_reference_check
                  WHERE {reference_id} IS NULL
              );
        """, [reference_value, original_value])

        n_updated = db.rowcount
        if n_updated > 0: applied_corrections.append( (attribute, original_value, reference_value) )
        total_updated += n_updated
        print( f"PTS | {attribute}: {original_value!r} -> {reference_value!r} ({n_updated} records updated)" )

print( f"\nPTS | total records updated: {total_updated}" )


# expected output:
# PTS | total records updated: 0

In [ ]:
# ------------------------------------------------------------
# validate reference correspondence after reconciliation
# ------------------------------------------------------------

n_unmatched_region, n_unmatched_institution, n_unmatched_diagnosis = \
db.sql("""
SELECT
  -- TODO | count number of unmatched records
  ...,
  ...,
  ...
FROM v_reference_check;
""").fetchone()

print(f"unmatched regions:      {n_unmatched_region}")
print(f"unmatched institutions: {n_unmatched_institution}")
print(f"unmatched diagnoses:    {n_unmatched_diagnosis}")

if any((n_unmatched_region, n_unmatched_institution, n_unmatched_diagnosis)):
    raise ValueError("PTS | unresolved reference correspondences")

print("reference validation successful")


# expected output:
# unmatched regions:      0
# unmatched institutions: 0
# unmatched diagnoses:    0
# reference validation successful

In [ ]:
# ------------------------------------------------------------
# record source-data corrections
# ------------------------------------------------------------

import pandas as pd
from pathlib import Path

dataset = "A"

output_file = Path("../03_output/sns_integration_check.csv")
columns = ["dataset", "attribute", "original_value", "reference_value"]

# load corrections previously recorded
if output_file.exists(): previous = pd.read_csv( output_file, dtype=str, keep_default_na=False )
else: previous = pd.DataFrame( columns=columns )

# retain corrections from other datasets
previous = previous[ previous["dataset"] != dataset ]

# record corrections applied to the current dataset
records = []
for attribute, original_value, reference_value in applied_corrections:
    records.append({
        "dataset": dataset,
        "attribute": attribute,
        "original_value": original_value,
        "reference_value": reference_value
    })
current = pd.DataFrame( records, columns=columns )

# combine and save
integration_check = pd.concat( [previous, current], ignore_index=True )
output_file.parent.mkdir( parents=True, exist_ok=True )
integration_check.to_csv( output_file, index=False )

print(f"dataset {dataset}: {len(current)} corrections recorded")
print(f"integration check saved to: {output_file}")


# expected output:
# dataset A: 0 corrections recorded
# integration check saved to: ../03_output/sns_integration_check.csv

In [ ]:
# ------------------------------------------------------------
# ensure all source records have valid reference correspondences
# ------------------------------------------------------------

# verify source records against reference tables
# notice the usage of "fetchall()" and "dict"
counts = dict( db.sql("""
SELECT 'source', COUNT(*)
FROM v_reference_check

UNION ALL

-- TODO | identify attributes
SELECT ..., ...
FROM v_reference_check
WHERE id_region IS NOT NULL
  AND id_institution IS NOT NULL
  AND id_diagnosis IS NOT NULL;
""").fetchall())

n_source_records = counts[ 'source' ]
n_matched_records = counts[ 'matched' ]
print(f"source records:  {n_source_records}")
print(f"matched records: {n_matched_records}")
assert n_source_records == n_matched_records, "PTS | integration aborted: unmatched source records"


# expected output:
# source records:  326692
# matched records: 326692

In [ ]:
# ------------------------------------------------------------
# integrate dataset into the relational database
# ------------------------------------------------------------

# idempotent integration:
# - repeated execution must produce the same logical dataset,
# - without accumulating duplicate records from previous runs

# delete previously integrated records from dataset_A (2021–2024)
# without affecting records from other periods (e.g., dataset_B)

# resolve foreign keys using the validated reference tables
# preserve all original records, including repeated observation units

# postgres_execute('pg', ...) executes SQL directly in PostgreSQL
# - allowing INSERT statements and PostgreSQL-specific functions
# - execute DELETE and INSERT together in PostgreSQL

db.sql("""
CALL postgres_execute('pg', $$
  DELETE FROM source_record
  WHERE period >= DATE '2021-01-01'
    AND period < DATE '2025-01-01';

  INSERT INTO source_record (
    period,
    id_institution,
    id_diagnosis,
    age_group,
    sex,
    num_inpatient_admission,
    num_inpatient_day,
    num_ambulatory_episode,
    num_death
  )
  SELECT
    TO_DATE(s.period || '-01', 'YYYY-MM-DD'),
    i.id,
    d.id,
    s.age_group,
    s.sex,
    s.num_inpatient_admission,
    s.num_inpatient_day,
    s.num_ambulatory_episode,
    s.num_death
  FROM source_stage s
  JOIN region r
    ON s.region = r.name
  JOIN institution i
    ON s.institution = i.name
   AND i.id_region = r.id
   -- TODO | identify JOIN
   ...
$$);
""")

# verify the number of integrated records
n_integrated_records = \
db.sql("""
SELECT COUNT(*)
FROM source_record
WHERE period >= DATE '2021-01-01'
  AND period < DATE '2025-01-01';
""").fetchone()[0]

print(f"integrated records: {n_integrated_records}")
assert n_integrated_records == n_source_records, "PTS | integration error: unexpected number of records"


# expected output:
# integrated records: 326692

In [ ]:
# ------------------------------------------------------------
# inspect repeated observation units
# ------------------------------------------------------------

# create a view with one row per observation unit
# identify conflicting hospital activity values
db.sql("""
CALL postgres_execute('pg', $$
  CREATE OR REPLACE VIEW v_observation_unit AS
  SELECT
    period,
    id_institution,
    id_diagnosis,
    age_group,
    sex,
    COUNT(*) AS n_records,
    COUNT( DISTINCT (
      num_death,
      num_inpatient_day,
      num_inpatient_admission,
      num_ambulatory_episode
    )) > 1 AS has_activity_conflict
  FROM source_record
  GROUP BY period, id_institution, id_diagnosis, age_group, sex
$$);
""")

# quantify repeated and conflicting observation units in dataset_A
db.sql("""
SELECT
  COUNT(*) AS n_observation_units,
  COUNT(*) FILTER ( WHERE n_records = 1 ) AS n_single_units,
  COUNT(*) FILTER ( WHERE n_records > 1 ) AS n_repeated_units,
  SUM( n_records - 1 ) AS n_excess_records,
  COUNT(*) FILTER ( WHERE has_activity_conflict ) AS n_conflicting_units
FROM v_observation_unit
WHERE period < DATE '2025-01-01';
""").show()

# inspect examples of repeated observation units
db.sql("""
SELECT *
FROM v_observation_unit
WHERE period < DATE '2025-01-01'
  AND n_records > 1
ORDER BY n_records DESC
LIMIT 5;
""").show()


# expected output (partial):
# ┌─────────────────────┬────────────────┬──────────────────┬──────────────────┬─────────────────────┐
# │ n_observation_units │ n_single_units │ n_repeated_units │ n_excess_records │ n_conflicting_units │
# │        int64        │     int64      │      int64       │      int128      │        int64        │
# ├─────────────────────┼────────────────┼──────────────────┼──────────────────┼─────────────────────┤
# │              317239 │         309555 │             7684 │             9453 │                7684 │


In [ ]:
# ------------------------------------------------------------
# analyze observation unit duplications in dataset
# ------------------------------------------------------------

# examine the distribution of records per observation unit
# n_duplicates represents the number of records beyond the first

db.sql("""
SELECT
  n_records - 1 AS n_duplicates,
  COUNT(*) AS n_observation_units
FROM v_observation_unit
WHERE period < DATE '2025-01-01'
GROUP BY n_records
ORDER BY n_records;
""").show()

# inspect the observation units with the most duplications
db.sql("""
SELECT
  n_records - 1 AS n_duplicates,
  has_activity_conflict,
  period,
  id_institution,
  id_diagnosis,
  age_group,
  sex
FROM v_observation_unit
WHERE period < DATE '2025-01-01'
  AND n_records > 1
ORDER BY n_duplicates DESC, period, id_institution, id_diagnosis, age_group, sex
LIMIT 10;
""").show()

In [ ]:
# ------------------------------------------------------------
# consolidate repeated observation units
# ------------------------------------------------------------

# alternative consolidation strategies:
#
# 1. Median aggregation
#    compute the median of each quantitative attribute;
#    simple, but may produce a record that never existed
#
# 2. Representative record
#    select the original record closest to the medians (L1);
#    preserves an actual record, but requires tie handling
#
# 3. Worst-case selection (ADOPTED)
#    select an original record using descending lexicographic
#    ordering: deaths, inpatient days, admissions, episodes;
#    simple, deterministic, and preserves an actual record
#
# this is an operational heuristic
# - not a guarantee of clinical severity or correctness
#
# the adopted rule prioritizes higher adverse-outcome counts
# - but does not necessarily identify greater clinical severity
#
# the same consolidation rule must be applied to dataset_A and dataset_B

# NOTE:
# the SNS was contacted by email to better understand why multiple records
# exist for the same observation unit, sometimes with conflicting values
# - so far, no response has been received
#
# consequently, the following consolidation strategies are considered
# - without assuming that repeated records are necessarily data errors

db.sql("""
CALL postgres_execute('pg', $$
DROP VIEW IF EXISTS v_consolidated_observation CASCADE;
$$);
""")

db.sql("""
CALL postgres_execute('pg', $$
  CREATE OR REPLACE VIEW v_consolidated_observation AS
  SELECT
    DISTINCT ON ( period, id_institution, id_diagnosis, age_group, sex )
    period,
    id_institution,
    id_diagnosis,
    age_group,
    sex,
    num_inpatient_admission,
    num_inpatient_day,
    num_ambulatory_episode,
    num_death
  FROM source_record
  ORDER BY
    period,
    id_institution,
    id_diagnosis,
    age_group,
    sex,
    -- TODO | implement the "worst-case selection" consolidation strategy
    ...
    id ASC
$$);
""")

# verify that consolidation produces one record per observation unit
db.sql("""
SELECT
  (SELECT COUNT(*) FROM source_record) AS n_source_records,
  (SELECT COUNT(*) FROM v_observation_unit) AS n_observation_units,
  (SELECT COUNT(*) FROM v_consolidated_observation) AS n_consolidated_observations;
""").show()


# expected output:
# ┌──────────────────┬─────────────────────┬─────────────────────────────┐
# │ n_source_records │ n_observation_units │ n_consolidated_observations │
# │      int64       │        int64        │            int64            │
# ├──────────────────┼─────────────────────┼─────────────────────────────┤
# │           326692 │              317239 │                      317239 │

In [ ]:
# ------------------------------------------------------------
# construct analytical observations and prediction targets
# ------------------------------------------------------------

# each consolidated observation defines three prediction targets:
#
# has_death          : binary classification (0 / 1)
# inpatient_level    : multiclass classification (low / medium / high)
# num_inpatient_day  : regression
#
# inpatient_ratio = inpatient admissions / (inpatient admissions + ambulatory episodes)
# inpatient_level is determined by the intervals in the reference table
#
# when the denominator is zero, inpatient_ratio is NULL;
# the LEFT JOIN preserves the observation with an undefined inpatient_level
#
# hospital activity counts are used to construct the targets,
# but are not available as features for the predicted month
db.sql("""
CALL postgres_execute('pg', $$
DROP VIEW IF EXISTS v_analytical_observation CASCADE;
$$);
""")

db.sql("""
CALL postgres_execute('pg', $$
  CREATE OR REPLACE VIEW v_analytical_observation AS
  SELECT
    o.period,
    o.id_institution,
    o.id_diagnosis,
    o.age_group,
    o.sex,
    
    l.name AS inpatient_level,
    o.num_inpatient_day,
    o.has_death
    
  FROM (
    SELECT
      period,
      id_institution,
      id_diagnosis,
      age_group,
      sex,
      
      num_inpatient_admission::NUMERIC / NULLIF( num_inpatient_admission + num_ambulatory_episode, 0  ) 
                                         AS inpatient_ratio,
      num_inpatient_day,
      -- TODO | implement the binary target (as a zero or one INTEGER)
      ...

    FROM v_consolidated_observation
  ) o
  LEFT JOIN inpatient_level l
  ON o.inpatient_ratio >= l.min_ratio
  AND ( o.inpatient_ratio < l.max_ratio OR (o.inpatient_ratio = 1 AND l.max_ratio = 1) )
$$);
""")

# inspect the resulting analytical observations
db.sql("""
SELECT *
FROM v_analytical_observation
ORDER BY period
LIMIT 10;
""").show()


# verify that analytical observations preserve the observation units
n_consolidated, n_analytical = \
db.sql("""
SELECT
  (SELECT COUNT(*) FROM v_consolidated_observation),
  (SELECT COUNT(*) FROM v_analytical_observation);
""").fetchone()
print(f"consolidated observations: {n_consolidated}")
print(f"analytical observations:   {n_analytical}")
assert n_consolidated == n_analytical, "PTS | analytical dataset: unexpected number of observations"


# expected output (partial):
# ┌────────────┬────────────────┬──────────────┬───────────┬─────────┬─────────────────┬───────────────────┬───────────┐
# │   period   │ id_institution │ id_diagnosis │ age_group │   sex   │ inpatient_level │ num_inpatient_day │ has_death │
# │    date    │     int32      │    int32     │  varchar  │ varchar │     varchar     │       int32       │   int32   │
# ├────────────┼────────────────┼──────────────┼───────────┼─────────┼─────────────────┼───────────────────┼───────────┤
# │ 2021-01-01 │              1 │            1 │ [0-1[     │ M       │ high            │                 7 │         0 │
# │ 2021-01-01 │              1 │            1 │ [15-25[   │ M       │ high            │                 1 │         0 |
# ...
# │ 2021-01-01 │              1 │            1 │ [65-120[  │ F       │ high            │               156 │         1 |
# ...
#
# consolidated observations: 317239
# analytical observations:   317239

In [ ]:
# ------------------------------------------------------------
# analyze undefined inpatient levels in dataset
# ------------------------------------------------------------

# quantify observations without a defined inpatient level
# and determine their proportion in the analytical dataset
db.sql("""
SELECT
  COUNT(*) AS n_observations,
  COUNT(*) FILTER ( WHERE inpatient_level IS NULL ) AS n_undefined,
  ROUND( 100.0 * COUNT(*) FILTER ( WHERE inpatient_level IS NULL ) / COUNT(*), 2 ) AS pct_undefined
FROM v_analytical_observation
WHERE period < DATE '2025-01-01';
""").show()


# verify how many observations have an undefined inpatient ratio
# due to the absence of both inpatient and ambulatory activity
db.sql("""
SELECT
  COUNT(*) AS n_zero_denominator
FROM v_consolidated_observation
WHERE period < DATE '2025-01-01'
  AND num_inpatient_admission + num_ambulatory_episode = 0;
""").show()

In [ ]:
# ------------------------------------------------------------
# analyze prediction target distributions in dataset
# ------------------------------------------------------------

# binary classification: distribution of death occurrence
db.sql("""
SELECT
  has_death,
  COUNT(*) AS n_observations
FROM v_analytical_observation
WHERE period < DATE '2025-01-01'
GROUP BY has_death
ORDER BY has_death;
""").show()

# multiclass classification: distribution of inpatient levels
db.sql("""
SELECT
  inpatient_level,
  -- TODO | implement distribution of inpatient levels
  ...
""").show()

# regression: descriptive statistics of inpatient days
db.sql("""
SELECT
  MIN( num_inpatient_day ) AS min_days,
  MAX( num_inpatient_day ) AS max_days,
  ROUND( AVG( num_inpatient_day ), 2 ) AS mean_days,
  MEDIAN( num_inpatient_day ) AS median_days
FROM v_analytical_observation
WHERE period < DATE '2025-01-01';
""").show()


# expected output (partial):
# ┌─────────────────┬────────────────┐
# │ inpatient_level │ n_observations │
# │     varchar     │     int64      │
# ├─────────────────┼────────────────┤
# │ high            │         173630 │
# │ low             │          60900 │
# │ medium          │          82709 │
# └─────────────────┴────────────────┘
#
# ──────────┬──────────┬───────────┬─────────────┐
# │ min_days │ max_days │ mean_days │ median_days │
# │  int32   │  int32   │  double   │   double    │
# ├──────────┼──────────┼───────────┼─────────────┤
# │        0 │    11078 │     83.45 │        15.0 │

### Extreme Values and Regression

Some observations contain unusually high numbers of inpatient days. These values represent aggregated hospital activity and should not automatically be considered data errors.

Such extreme values may influence model training and regression performance, particularly when evaluating prediction errors.

**TODO | Investigate and discuss:**

- identify and characterize observations with unusually high `num_inpatient_day` values
- examine the characteristics of these observations and identify possible patterns across institutions, diagnoses, and age groups.
- investigate their potential impact on regression modelling and evaluation
- discuss and justify any decisions made to address these observations

**Important:** preserve the original analytical-dataset.

In [ ]:
# ------------------------------------------------------------
# inspect observations with approximately N (+/- tolerance%) inpatient days
# ------------------------------------------------------------

N = 11000
tolerance = 0.05  # +/- 5%

db.sql(f"""
SELECT
  o.period,
  i.name AS institution,
  d.description AS diagnosis,
  o.age_group,
  o.sex,
  o.num_inpatient_admission,
  o.num_inpatient_day,
  o.num_ambulatory_episode,
  o.num_death
FROM v_consolidated_observation o
JOIN institution i ON o.id_institution = i.id
JOIN diagnosis d ON o.id_diagnosis = d.id
WHERE o.num_inpatient_day BETWEEN {N * (1 - tolerance)} AND {N * (1 + tolerance)}
ORDER BY ABS(o.num_inpatient_day - {N}), o.period
LIMIT 20;
""").show()

In [ ]:
# ------------------------------------------------------------
# inspect extreme inpatient-day values
# ------------------------------------------------------------

import numpy as np

# retrieve consolidated observations from dataset_A
data = db.sql("""
SELECT
  period,
  id_institution,
  id_diagnosis,
  age_group,
  sex,
  num_inpatient_admission,
  num_inpatient_day
FROM v_consolidated_observation
WHERE period < DATE '2025-01-01';
""").df()

# examine the target distribution
target = "num_inpatient_day"

# quantile: proportion of observations below a given value (0 to 1)
# percentile: the same concept expressed as a percentage (0 to 100)
# e.g., quantile 0.99 = 99th percentile: 99% of values are at or below it
percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
quantile = 0.99

display( data[ target ].describe( percentiles=percentiles ).round(2) )

# identify candidate outliers using the 99th percentile
threshold = data[ target ].quantile( quantile )
outliers = data[ data[ target ] > threshold ]

print()
print(f"candidate outliers: {len(outliers)}")
print(f"threshold: {threshold:.2f}")
print()
# inspect the observations with the highest values
display( outliers.sort_values(target, ascending=False).head(20) )

# illustrate a logarithmic target transformation
data["log_inpatient_day"] = np.log1p(data[target])
print()
display( data[[target, "log_inpatient_day"]].describe( percentiles=percentiles ).round(2) )

In [ ]:
# ------------------------------------------------------------
# explore categorical feature–target relationships
# ------------------------------------------------------------

# construct a contingency matrix:
# - rows represent feature values
# - columns represent target classes
# - cells contain observation frequencies

feature = "id_diagnosis"
target = "inpatient_level"

matrix = db.sql(f"""
  PIVOT (
    SELECT "{feature}", "{target}"
    FROM v_analytical_observation
    WHERE period < DATE '2025-01-01' AND "{target}" IS NOT NULL
  )
  ON "{target}"
  USING COUNT(*)
  GROUP BY "{feature}"
  ORDER BY "{feature}"
""").df().set_index(feature)

display(matrix)

In [ ]:
# ------------------------------------------------------------
# derive classification rule from the contingency matrix
# ------------------------------------------------------------

# sort classes alphabetically for deterministic tie-breaking,
# regardless of the column order returned by PIVOT
matrix = matrix.reindex( columns=sorted(matrix.columns) )
# predict the majority class for each feature value
rule = matrix.idxmax( axis=1 ).to_frame( "y_pred" )

# count observations, correct predictions, and errors
rule["n_observations"] = matrix.sum(axis=1)
rule["n_correct"] = matrix.max(axis=1)
rule["n_errors"] = rule["n_observations"] - rule["n_correct"]
display(rule)

# compute the overall classification error for this feature
error = rule["n_errors"].sum() / rule["n_observations"].sum()

print(f"training classification error ({feature}): {error:.4f}")

In [ ]:
# ------------------------------------------------------------
# temporal cross-validation with an expanding training window
# ------------------------------------------------------------

import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.dummy import DummyClassifier

# use the observations currently available
# from Datasets A and B; Dataset C remains excluded

data = db.sql("""
SELECT
  period,
  id_institution,
  id_diagnosis,
  age_group,
  sex,
  EXTRACT( MONTH FROM period )::INTEGER AS month,
  inpatient_level
FROM v_analytical_observation
WHERE period < DATE '2026-07-01'
  AND inpatient_level IS NOT NULL
ORDER BY period;
""").df()

X = data.drop(columns=["period", "inpatient_level"])
y = data["inpatient_level"]

# ------------------------------------------------------------
# define temporal cross-validation folds
# ------------------------------------------------------------

# ------------------------------------------------------------
# IMPORTANT:
# conventional K-fold cross-validation does not enforce
# chronological ordering between training and testing
# - each fold is used once for testing, with the remaining K-1 for training
# - folds may be shuffled or stratified to preserve class proportions
#
# temporal cross-validation trains exclusively on past data
# and evaluates predictions on subsequent periods
# ------------------------------------------------------------

# ------------------------------------------------------------
# automatically identify the years available in the dataset
# use an expanding training window:
#
# fold 1: train on 2021             | test on 2022
# fold 2: train on 2021-2022        | test on 2023
# fold 3: train on 2021-2023        | test on 2024
# fold 4: train on 2021-2024        | test on 2025
# fold 5: train on 2021-2025        | test on 2026 (Jan-Jun)
#
# only folds with available observations are constructed
# each model is trained exclusively on years preceding its test year
# the latest test year may contain only partially integrated data
# ------------------------------------------------------------

available_years = sorted( data["period"].dt.year.unique() )
cv_folds = []
test_years = []

# available_years[1:] skips the first year, used for initial training
# - each subsequent year is a test fold, trained on all previous years
# - therefore, at least two years are required
for year in available_years[ 1: ]:
  train_idx = data.index[data["period"].dt.year < year].to_numpy()
  test_idx = data.index[data["period"].dt.year == year].to_numpy()

  cv_folds.append((train_idx, test_idx))
  test_years.append(year)


# ------------------------------------------------------------
# evaluate a baseline classifier using the available folds
# ------------------------------------------------------------
# require at least one temporal cross-validation fold
if not cv_folds: raise ValueError("PTS | temporal cross-validation requires at least two years")

model = DummyClassifier( strategy="most_frequent" )
scores = cross_validate( model, X, y, cv=cv_folds,
                         scoring=["accuracy", "balanced_accuracy", "f1_macro"] )

results = pd.DataFrame({
  "test_year": test_years,
  "accuracy": scores["test_accuracy"],
  "balanced_accuracy": scores["test_balanced_accuracy"],
  "f1_macro": scores["test_f1_macro"] })

display(results.round(4))

In [ ]:
# ------------------------------------------------------------
# save and load multiple predictive models
# ------------------------------------------------------------

import joblib
from sklearn.dummy import DummyClassifier, DummyRegressor

# prepare illustrative training data from dataset
data = db.sql("""
SELECT
  id_institution,
  id_diagnosis,
  age_group,
  sex,
  EXTRACT(MONTH FROM period)::INTEGER AS month,
  inpatient_level,
  num_inpatient_day,
  has_death
FROM v_analytical_observation
WHERE period < DATE '2025-01-01'
  AND inpatient_level IS NOT NULL
  AND num_inpatient_day IS NOT NULL
  AND has_death IS NOT NULL;
""").df()

# define features and prediction targets
targets = ["inpatient_level", "num_inpatient_day", "has_death"]
X = data.drop( columns=targets )

# create one illustrative model per prediction problem
models = {
  "inpatient_level": DummyClassifier(strategy="most_frequent"),
  "num_inpatient_day": DummyRegressor(strategy="mean"),
  "has_death": DummyClassifier(strategy="most_frequent")
}

# train the models
for target, model in models.items():
  model.fit( X, data[target] )

# ------------------------------------------------------------
# persist all models in a single joblib file
# ------------------------------------------------------------
output_dir = Path("../03_output")
output_dir.mkdir( parents=True, exist_ok=True )

model_file = output_dir / "_mLDm_example_models.joblib"
joblib.dump( models, model_file )

print(f"Models saved to: {model_file}")

# ------------------------------------------------------------
# load and apply the persisted models without retraining
# ------------------------------------------------------------
deployed_models = joblib.load( model_file )

# illustrate prediction on a few existing observations
new_data = X.head(5).copy()

for target, model in deployed_models.items():
  new_data[ target ] = model.predict( new_data[X.columns] )

display(new_data)